# UW-Madison GI Tract Image Segmentation  
## 2.5D EfficientNet-B0 U-Net Training Pipeline

This notebook builds a complete training pipeline for multi-organ MRI segmentation in the UW-Madison GI Tract Image Segmentation dataset.

The model uses a 2.5D input representation, where three neighboring MRI slices are stacked as input channels. The segmentation network predicts three binary masks, one for each target organ.

Core model design:

```text
2.5D MRI input: [slice n-2, slice n, slice n+2]
→ EfficientNet-B0 encoder
→ U-Net decoder
→ 3-channel segmentation output
→ large_bowel, small_bowel, stomach
```

Main workflow:

1. Load the original annotation CSV.
2. Parse image metadata from file paths.
3. Merge metadata with RLE annotations.
4. Decode RLE masks and verify mask alignment.
5. Build 2.5D image inputs from neighboring slices.
6. Create case-wise GroupKFold splits.
7. Define the PyTorch Dataset and DataLoader.
8. Train an EfficientNet-B0 U-Net model.
9. Evaluate validation Dice score during training.
10. Save the best checkpoint and training history.


## 0. Imports and Project Configuration

This section imports the basic preprocessing libraries and defines the shared configuration used throughout the notebook.


In [ ]:
import os
import re
import glob
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold

pd.set_option("display.max_columns", 100)


In [ ]:
class CFG:
    seed = 42

    # Candidate data directories used in different Kaggle environments.
    # The mounted input path may differ between competition and notebook contexts.
    data_dir_candidates = [
        Path("/kaggle/input/uw-madison-gi-tract-image-segmentation"),
        Path("/kaggle/input/competitions/uw-madison-gi-tract-image-segmentation"),
    ]

    data_dir = None
    for candidate in data_dir_candidates:
        if candidate.exists():
            data_dir = candidate
            break

    if data_dir is None:
        # Fallback path for a standard Kaggle competition dataset mount.
        data_dir = Path("/kaggle/input/uw-madison-gi-tract-image-segmentation")

    train_csv = data_dir / "train.csv"
    train_img_dir = data_dir / "train"
    test_img_dir = data_dir / "test"

    classes = ["large_bowel", "small_bowel", "stomach"]

    # 2.5D input configuration.
    # A stride of 2 uses neighboring slices [n-2, n, n+2].
    slice_stride = 2

    # Input resolution used for both images and masks.
    # 456 is EfficientNet-B5's native training resolution.
    # (Use 384 when training the smaller EfficientNet-B0.)
    img_size = 456

    n_folds = 5

print("Data directory:", CFG.data_dir)
print("Train CSV:", CFG.train_csv)
print("Train image directory:", CFG.train_img_dir)


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.seed)


## 1. Load the Original Training Annotations

The original `train.csv` file stores segmentation labels in long format. Each image appears once per organ class:

- `large_bowel`
- `small_bowel`
- `stomach`

For training, the annotations are converted into wide format so that each MRI slice is represented by one row with three RLE mask columns.


In [ ]:
train_raw = pd.read_csv(CFG.train_csv)

print("Raw train.csv shape:", train_raw.shape)
train_raw.head()


In [ ]:
train_anno = train_raw.pivot(
    index="id",
    columns="class",
    values="segmentation"
).reset_index()

# Make sure all target organ columns are present.
for c in CFG.classes:
    if c not in train_anno.columns:
        train_anno[c] = np.nan

# Keep the annotation columns in a fixed order.
train_anno = train_anno[["id"] + CFG.classes]

# Represent missing masks as empty RLE strings.
train_anno[CFG.classes] = train_anno[CFG.classes].fillna("")

print("Wide annotation shape:", train_anno.shape)
train_anno.head()


## 2. Scan Image Paths and Parse Metadata

Each training image filename contains useful metadata such as slice index, image size, and pixel spacing.

Example filename:

```text
slice_0001_266_266_1.50_1.50.png
```

The folder structure provides the case ID and day ID. These fields are parsed and later used for grouping, fold splitting, and 2.5D slice selection.


In [ ]:
train_image_paths = sorted(
    glob.glob(str(CFG.train_img_dir / "case*" / "case*_day*" / "scans" / "*.png"))
)

print("Number of train images:", len(train_image_paths))

if len(train_image_paths) > 0:
    print("Example image path:")
    print(train_image_paths[0])


In [ ]:
def parse_image_path(path):
    """
    Parse metadata from one image path.

    Expected structure:
        train/case123/case123_day20/scans/slice_0001_266_266_1.50_1.50.png

    Returns
    -------
    dict with:
        id, case, day, slice, height, width, spacing_x, spacing_y, image_path
    """
    path = Path(path)
    filename = path.stem

    # Example filename stem:
    # slice_0001_266_266_1.50_1.50
    parts = filename.split("_")
    if len(parts) < 6:
        raise ValueError(f"Unexpected filename format: {filename}")

    slice_id = int(parts[1])
    height = int(parts[2])
    width = int(parts[3])
    spacing_x = float(parts[4])
    spacing_y = float(parts[5])

    # Folder hierarchy:
    # path.parents[0] = scans
    # path.parents[1] = case123_day20
    # path.parents[2] = case123
    case_str = path.parents[2].name
    case_day_str = path.parents[1].name

    case_match = re.search(r"case(\d+)", case_str)
    day_match = re.search(r"case(\d+)_day(\d+)", case_day_str)

    if case_match is None or day_match is None:
        raise ValueError(f"Unexpected folder format: {path}")

    case = int(case_match.group(1))
    day = int(day_match.group(2))

    # Image ID format used in train.csv:
    # case123_day20_slice_0001
    image_id = f"case{case}_day{day}_slice_{slice_id:04d}"

    return {
        "id": image_id,
        "case": case,
        "day": day,
        "slice": slice_id,
        "height": height,
        "width": width,
        "spacing_x": spacing_x,
        "spacing_y": spacing_y,
        "image_path": str(path),
    }


In [ ]:
meta_df = pd.DataFrame([parse_image_path(p) for p in train_image_paths])

print("Metadata shape:", meta_df.shape)
meta_df.head()


## 3. Merge Image Metadata with RLE Annotations

The parsed image metadata is merged with the wide annotation table. After this step, each row corresponds to one MRI slice and contains:

- image path and metadata
- case, day, and slice information
- RLE masks for the three organ classes


In [ ]:
df = meta_df.merge(train_anno, on="id", how="left")

# Treat missing annotations as empty masks.
df[CFG.classes] = df[CFG.classes].fillna("")

print("Merged dataframe shape:", df.shape)
df.head()


In [ ]:
for c in CFG.classes:
    df[f"has_{c}"] = df[c].apply(lambda x: 1 if isinstance(x, str) and len(x) > 0 else 0)

df["has_any"] = df[[f"has_{c}" for c in CFG.classes]].max(axis=1)

summary = df[[f"has_{c}" for c in CFG.classes] + ["has_any"]].mean().to_frame("positive_ratio")
summary


## 4. RLE Mask Decoding and Encoding

The competition labels are stored as run-length encoded strings. During training, each RLE string is decoded into a binary mask.

This implementation uses C-order indexing for reshaping decoded masks:

```python
mask.reshape((height, width), order="C")
```

The same ordering is used when converting predicted masks back to RLE format:

```python
mask.flatten(order="C")
```


In [ ]:
def rle_decode(mask_rle, shape):
    """
    Decode an RLE string into a binary mask using C-order reshape.

    Parameters
    ----------
    mask_rle : str
        Run-length encoded mask.
    shape : tuple
        (height, width)

    Returns
    -------
    mask : np.ndarray
        Binary mask with shape (height, width).

    Notes
    -----
    This project uses C-order reshape based on visual verification.
    The previous F-order version caused ground-truth masks to be misaligned.
    """
    height, width = shape
    mask = np.zeros(height * width, dtype=np.uint8)

    if mask_rle is None or mask_rle == "" or pd.isna(mask_rle):
        return mask.reshape((height, width), order="C")

    s = list(map(int, str(mask_rle).split()))
    starts = np.array(s[0::2]) - 1
    lengths = np.array(s[1::2])
    ends = starts + lengths

    for start, end in zip(starts, ends):
        mask[start:end] = 1

    return mask.reshape((height, width), order="C")


def rle_encode(mask):
    """
    Encode a binary mask into an RLE string using C-order flattening.

    Parameters
    ----------
    mask : np.ndarray
        Binary mask with shape (height, width).

    Returns
    -------
    rle : str
        Run-length encoded mask.
    """
    pixels = mask.astype(np.uint8).flatten(order="C")
    pixels = np.concatenate([[0], pixels, [0]])

    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]

    return " ".join(str(x) for x in runs)


In [ ]:
# Check RLE decoding on one sample with at least one positive mask.
sample_row = df[df["has_any"] == 1].iloc[0]

h, w = int(sample_row["height"]), int(sample_row["width"])

decoded_masks = []
for c in CFG.classes:
    mask = rle_decode(sample_row[c], shape=(h, w))
    decoded_masks.append(mask)
    print(c, mask.shape, mask.sum())

decoded_masks = np.stack(decoded_masks, axis=-1)
print("Stacked mask shape:", decoded_masks.shape)


## 4.1 Ground-truth Mask Verification

Before model training, decoded masks are visualized on top of the raw MRI slices. This check confirms that the RLE decoding, mask orientation, and image alignment are correct.

This verification uses only the raw image and decoded ground-truth masks. It does not depend on the Dataset class, data augmentation, or model inference.


In [ ]:
def normalize_for_display(img):
    """
    Normalize a raw MRI image to 0-1 for visualization.
    """
    img = img.astype(np.float32)
    p1 = np.percentile(img, 1)
    p99 = np.percentile(img, 99)
    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-6)
    return img


def visualize_raw_ground_truth(row):
    """
    Visualize raw image and C-order decoded ground-truth masks.
    """
    img = cv2.imread(str(row["image_path"]), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {row['image_path']}")

    img_norm = normalize_for_display(img)
    h, w = img.shape[:2]

    print("=" * 100)
    print("Raw ground-truth verification")
    print("ID:", row["id"])
    print("Image path:", row["image_path"])
    print("Image shape:", img.shape)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for i, c in enumerate(CFG.classes):
        mask = rle_decode(row[c], shape=(h, w))

        print(f"{c}: mask_area={mask.sum()}")

        axes[i].imshow(img_norm, cmap="gray")
        axes[i].imshow(
            np.ma.masked_where(mask == 0, mask),
            cmap="Reds",
            alpha=0.55,
            interpolation="nearest",
        )
        axes[i].set_title(f"Raw GT: {c}")
        axes[i].axis("off")

    plt.suptitle(f"C-order decoded raw GT | {row['id']}", fontsize=14)
    plt.tight_layout()
    plt.show()


# Visualize representative samples that contain at least one organ mask.
positive_examples = df[df["has_any"] == 1].sample(
    n=min(3, int(df["has_any"].sum())),
    random_state=CFG.seed,
)

for _, row in positive_examples.iterrows():
    visualize_raw_ground_truth(row)


## 5. MRI Image Reading and Percentile Normalization

MRI images are grayscale medical images rather than standard RGB images. To make the intensity range more stable, each slice is normalized with percentile clipping:

```text
clip intensities to the 1st and 99th percentiles
normalize the clipped image to the range [0, 1]
```

This reduces the influence of extreme pixel values and provides more consistent inputs for training.


In [ ]:
def read_image(path):
    """
    Read a single grayscale MRI slice and normalize it to 0-1.
    """
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)

    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")

    img = img.astype(np.float32)

    p1 = np.percentile(img, 1)
    p99 = np.percentile(img, 99)

    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-6)

    return img


In [ ]:
img = read_image(sample_row["image_path"])

print(img.shape, img.min(), img.max(), img.dtype)

plt.figure(figsize=(5, 5))
plt.imshow(img, cmap="gray")
plt.title("Normalized MRI Slice")
plt.axis("off")
plt.show()


## 6. Build 2.5D Input Paths

The model uses a 2.5D representation by stacking neighboring slices as three input channels.

For the main experiment, each input is constructed as:

```text
[slice n-2, slice n, slice n+2]
```

This allows a 2D U-Net model to use limited through-plane context while keeping the training pipeline simple and efficient.


In [ ]:
# Dictionary key: (case, day, slice)
# Dictionary value: image path
path_dict = {
    (int(row["case"]), int(row["day"]), int(row["slice"])): row["image_path"]
    for _, row in df.iterrows()
}

# Available slices for each case/day group, used for boundary handling.
slice_dict = (
    df.groupby(["case", "day"])["slice"]
    .apply(lambda x: sorted(x.astype(int).tolist()))
    .to_dict()
)

print("Number of case/day groups:", len(slice_dict))


In [ ]:
def get_nearest_slice(available_slices, target_slice):
    """
    If the target slice does not exist, return the nearest available slice.
    """
    available_slices = np.array(available_slices)
    nearest_idx = np.argmin(np.abs(available_slices - target_slice))
    return int(available_slices[nearest_idx])


def get_2_5d_paths(row, stride=2):
    """
    Return image paths for [n-stride, n, n+stride].

    Boundary slices are handled by selecting the nearest available slice.
    """
    case = int(row["case"])
    day = int(row["day"])
    center_slice = int(row["slice"])

    available_slices = slice_dict[(case, day)]

    target_slices = [
        center_slice - stride,
        center_slice,
        center_slice + stride,
    ]

    selected_paths = []
    selected_slices = []

    for s in target_slices:
        nearest_s = get_nearest_slice(available_slices, s)
        selected_slices.append(nearest_s)
        selected_paths.append(path_dict[(case, day, nearest_s)])

    return selected_paths, selected_slices


In [ ]:
prev_paths = []
center_paths = []
next_paths = []

prev_slices = []
center_slices = []
next_slices = []

for _, row in df.iterrows():
    paths, slices = get_2_5d_paths(row, stride=CFG.slice_stride)

    prev_paths.append(paths[0])
    center_paths.append(paths[1])
    next_paths.append(paths[2])

    prev_slices.append(slices[0])
    center_slices.append(slices[1])
    next_slices.append(slices[2])

df["image_path_prev"] = prev_paths
df["image_path_center"] = center_paths
df["image_path_next"] = next_paths

df["slice_prev"] = prev_slices
df["slice_center"] = center_slices
df["slice_next"] = next_slices

df[
    [
        "id", "case", "day", "slice",
        "slice_prev", "slice_center", "slice_next",
        "image_path_prev", "image_path_center", "image_path_next",
    ]
].head()


## 7. Read 2.5D Images

Each model input contains three grayscale MRI slices stacked along the channel dimension:

```text
channel 0 = slice n-2
channel 1 = slice n
channel 2 = slice n+2
```

Although the tensor has three channels, it is not an RGB image. The channels represent spatially adjacent MRI slices.


In [ ]:
def read_2_5d_image(row):
    """
    Read [n-2, n, n+2] slices and stack them as a 3-channel image.

    All neighboring slices are resized to the center slice size before stacking.
    """
    paths = [
        row["image_path_prev"],
        row["image_path_center"],
        row["image_path_next"],
    ]

    center_img = read_image(row["image_path_center"])
    center_h, center_w = center_img.shape[:2]

    images = []
    for p in paths:
        img = read_image(p)

        if img.shape[:2] != (center_h, center_w):
            img = cv2.resize(
                img,
                (center_w, center_h),
                interpolation=cv2.INTER_LINEAR,
            )

        images.append(img)

    image = np.stack(images, axis=-1)
    return image.astype(np.float32)


In [ ]:
# Select a sample after the 2.5D path columns are created.
sample_row = df[df["has_any"] == 1].iloc[0]
sample_img_25d = read_2_5d_image(sample_row)

print(sample_img_25d.shape, sample_img_25d.min(), sample_img_25d.max(), sample_img_25d.dtype)


## 8. Decode Three-channel Segmentation Masks

For each center slice, three binary masks are decoded from RLE format:

```text
mask[:, :, 0] = large_bowel
mask[:, :, 1] = small_bowel
mask[:, :, 2] = stomach
```

The resulting mask has shape:

```text
H × W × 3
```


In [ ]:
def read_mask(row):
    """
    Decode RLE masks for three organs.

    The mask shape is taken from the actual center image shape
    to avoid height/width mismatch.
    """
    center_img = cv2.imread(str(row["image_path_center"]), cv2.IMREAD_UNCHANGED)

    if center_img is None:
        raise FileNotFoundError(f"Cannot read center image: {row['image_path_center']}")

    height, width = center_img.shape[:2]

    masks = []
    for c in CFG.classes:
        mask = rle_decode(row[c], shape=(height, width))
        masks.append(mask)

    mask = np.stack(masks, axis=-1)
    return mask.astype(np.float32)


In [ ]:
sample_mask = read_mask(sample_row)

print(sample_mask.shape, sample_mask.dtype)
for i, c in enumerate(CFG.classes):
    print(c, sample_mask[:, :, i].sum())


## 9. Pre-training Visualization Check

Before training, representative samples are visualized to confirm the preprocessing pipeline.

The visualization checks that:

1. The 2.5D image channels are loaded correctly.
2. The three organ masks are decoded correctly.
3. The mask orientation matches the MRI slice.
4. The selected neighboring slices are consistent with the center slice.


In [ ]:
def visualize_sample(row):
    image = read_2_5d_image(row)
    mask = read_mask(row)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    channel_titles = [
        f"slice {row['slice_prev']}",
        f"slice {row['slice_center']}",
        f"slice {row['slice_next']}",
    ]

    for i in range(3):
        axes[0, i].imshow(image[:, :, i], cmap="gray")
        axes[0, i].set_title(channel_titles[i])
        axes[0, i].axis("off")

    for i, c in enumerate(CFG.classes):
        axes[1, i].imshow(image[:, :, 1], cmap="gray")
        axes[1, i].imshow(
            np.ma.masked_where(mask[:, :, i] == 0, mask[:, :, i]),
            cmap="Reds",
            alpha=0.55,
            interpolation="nearest",
        )
        axes[1, i].set_title(c)
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
sample_row = df[df["has_any"] == 1].iloc[0]
visualize_sample(sample_row)


## 10. Create Case-wise GroupKFold Splits

The dataset is split by case rather than by individual slice.

This is important because slices from the same case are highly correlated. A random slice-level split could place nearly identical slices in both training and validation sets, leading to data leakage and overly optimistic validation results.


In [ ]:
gkf = GroupKFold(n_splits=CFG.n_folds)

df["fold"] = -1

for fold, (_, valid_idx) in enumerate(gkf.split(df, groups=df["case"])):
    df.loc[valid_idx, "fold"] = fold

df["fold"].value_counts().sort_index()


In [ ]:
for fold in range(CFG.n_folds):
    train_cases = set(df[df["fold"] != fold]["case"].unique())
    valid_cases = set(df[df["fold"] == fold]["case"].unique())

    overlap = train_cases.intersection(valid_cases)

    print(f"Fold {fold}:")
    print("  Train cases:", len(train_cases))
    print("  Valid cases:", len(valid_cases))
    print("  Overlap:", len(overlap))


## 11. Save the Preprocessed DataFrame

The final preprocessing table contains image metadata, RLE masks, positive-mask indicators, 2.5D image paths, neighboring slice IDs, and fold assignments.

This table is saved as a CSV file and then used by the PyTorch Dataset in the training pipeline.


In [ ]:
save_path = Path("./train_preprocessed_2_5d.csv")
df.to_csv(save_path, index=False)

print("Saved to:", save_path)
print("Final dataframe shape:", df.shape)

df.head()


In [ ]:
df.columns.tolist()


## Preprocessing Output

After preprocessing, the notebook saves:

```text
train_preprocessed_2_5d.csv
```

Important columns include:

```text
id
case
day
slice
height
width
spacing_x
spacing_y
image_path
large_bowel
small_bowel
stomach
has_large_bowel
has_small_bowel
has_stomach
has_any
image_path_prev
image_path_center
image_path_next
slice_prev
slice_center
slice_next
fold
```

The training Dataset uses this table to return tensors in the following format:

```text
image: [3, 384, 384]
mask:  [3, 384, 384]
```


# Part 2. Model Training Pipeline

This section trains the 2.5D EfficientNet-B0 U-Net segmentation model.

Model design:

```text
Input:
    2.5D MRI image with 3 channels: [slice n-2, slice n, slice n+2]

Model:
    U-Net architecture with an EfficientNet-B0 encoder

Output:
    3 segmentation channels:
        channel 0 = large_bowel
        channel 1 = small_bowel
        channel 2 = stomach

Loss:
    BCEWithLogitsLoss + 3 × DiceLoss

Validation:
    Mean Dice score and per-class Dice score
```

The default configuration trains fold 0. The same pipeline can be repeated for additional folds if cross-validation ensembling is needed.


## 12. Package Installation

This cell installs `segmentation_models_pytorch` and `timm`, which are required to build the EfficientNet-B0 U-Net model.

On Kaggle, package availability can vary by environment. If the packages are already available, this installation step can be skipped.


In [ ]:
# Install model dependencies if they are not already available.
!pip install -q segmentation-models-pytorch timm


## 13. Training Imports and Configuration

This section imports the PyTorch, Albumentations, and segmentation model libraries used in the training stage.


In [ ]:
import time
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

try:
    import segmentation_models_pytorch as smp
except ImportError as e:
    raise ImportError(
        "segmentation_models_pytorch is not installed. "
        "Please uncomment and run the package installation cell above."
    ) from e

from tqdm.auto import tqdm


In [ ]:
# Training configuration.

# ------------------------------------------------------------------
# Smoke test switch.
# True  -> train 1 epoch on a small random subsample, just to verify the
#          whole pipeline (B5 + 456 + aux head + grad accumulation) runs
#          end to end and saves a checkpoint, in a few minutes.
# False -> real training (full data, full epochs).
# ------------------------------------------------------------------
CFG.smoke_test = False

CFG.fold = 0
CFG.epochs = 1 if CFG.smoke_test else 20
CFG.lr = 1e-4
CFG.weight_decay = 1e-2
CFG.eta_min = 1e-6
CFG.num_workers = 2
CFG.amp = True

# ------------------------------------------------------------------
# Wall-clock safety budget.
# Kaggle GPU sessions are killed at a hard 12h limit. In "Save & Run
# All" (Commit) mode a run that hits that wall FAILS and its
# /kaggle/working outputs are discarded -- including any checkpoint
# written mid-run. So we stop *ourselves* before the wall (with margin
# for one more epoch + validation + save), end cleanly, and persist.
# Re-running the notebook then resumes from the `last` checkpoint.
# ------------------------------------------------------------------
CFG.max_train_hours = 11.0   # per-session budget; each Kaggle session gets a fresh 12h
CFG.resume = True            # if a `last` checkpoint exists, continue from it

# ------------------------------------------------------------------
# Encoder / model scale.
# Larger encoders (b4/b5/...) usually improve accuracy but need more GPU
# memory and time. IMPORTANT: gradient accumulation only saves memory if
# batch_size is small. The per-step memory peak is set by batch_size, NOT
# by the effective batch. For B5 @ 456 on a 16GB GPU, keep batch_size tiny
# and recover the effective batch with accum_steps.
# ------------------------------------------------------------------
CFG.encoder_name = "efficientnet-b5"   # use "efficientnet-b0" for the baseline
CFG.encoder_weights = "imagenet"       # needs internet during training to fetch timm weights
CFG.dice_weight = 3.0

CFG.batch_size = 2     # small per-step batch so B5 @ 456 fits in memory
CFG.accum_steps = 4    # effective batch = batch_size * accum_steps = 8

# ------------------------------------------------------------------
# Auxiliary classification head (presence detection / classification gate).
# Enabled for this experiment.
# ------------------------------------------------------------------
CFG.use_aux_cls = True
CFG.cls_weight = 0.3   # weight of the classification BCE term in the total loss

# Short tag used in output filenames, derived from the encoder name.
# e.g. "efficientnet-b0" -> "effnetb0", "efficientnet-b5" -> "effnetb5"
CFG.model_tag = CFG.encoder_name.replace("efficientnet-", "effnet").replace("-", "")

CFG.device = "cuda" if torch.cuda.is_available() else "cpu"

print("Smoke test:", CFG.smoke_test)
print("Device:", CFG.device)
print("Encoder:", CFG.encoder_name)
print("Image size:", CFG.img_size)
print("Batch size:", CFG.batch_size)
print("Accum steps:", CFG.accum_steps,
      "(effective batch =", CFG.batch_size * CFG.accum_steps, ")")
print("Aux classification head:", CFG.use_aux_cls)
print("Epochs:", CFG.epochs)
print("Fold:", CFG.fold)
print("Max train hours:", CFG.max_train_hours)
print("Resume:", CFG.resume)


## 14. Data Augmentation Transforms

The training pipeline applies moderate augmentations to improve generalization:

- resize to the configured training resolution
- horizontal flip
- small translation, scale, and rotation
- random brightness and contrast adjustment

Masks are resized with nearest-neighbor interpolation to preserve binary label values.


In [ ]:
def get_train_transforms():
    return A.Compose([
        A.Resize(
            CFG.img_size,
            CFG.img_size,
            interpolation=cv2.INTER_LINEAR,
            mask_interpolation=cv2.INTER_NEAREST,
        ),
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=(-0.05, 0.05),
            scale=(0.9, 1.1),
            rotate=(-10, 10),
            p=0.5,
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.10,
            contrast_limit=0.10,
            p=0.3,
        ),
        ToTensorV2(transpose_mask=True),
    ])


def get_valid_transforms():
    return A.Compose([
        A.Resize(
            CFG.img_size,
            CFG.img_size,
            interpolation=cv2.INTER_LINEAR,
            mask_interpolation=cv2.INTER_NEAREST,
        ),
        ToTensorV2(transpose_mask=True),
    ])


## 15. PyTorch Dataset

The Dataset reads a row from the preprocessed DataFrame and returns one image-mask pair.

Returned tensors:

```text
image: torch.float32 tensor with shape [3, 384, 384]
mask:  torch.float32 tensor with shape [3, 384, 384]
```

The image is the 2.5D input `[slice n-2, slice n, slice n+2]`. The mask contains three organ channels:

```text
0 = large_bowel
1 = small_bowel
2 = stomach
```


In [ ]:
class UWGIDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = read_2_5d_image(row)
        mask = read_mask(row)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask = augmented["mask"]
        else:
            image = cv2.resize(
                image,
                (CFG.img_size, CFG.img_size),
                interpolation=cv2.INTER_LINEAR,
            )
            mask = cv2.resize(
                mask,
                (CFG.img_size, CFG.img_size),
                interpolation=cv2.INTER_NEAREST,
            )

            image = torch.from_numpy(image.transpose(2, 0, 1)).float()
            mask = torch.from_numpy(mask.transpose(2, 0, 1)).float()

        return image.float(), mask.float()


In [ ]:
# Check one transformed image-mask pair before training.
train_tmp = UWGIDataset(df[df["fold"] != CFG.fold].head(8), transform=get_train_transforms())

image, mask = train_tmp[0]
print("Image shape:", image.shape, image.dtype, image.min().item(), image.max().item())
print("Mask shape:", mask.shape, mask.dtype, mask.min().item(), mask.max().item())


## 16. DataLoader

The training and validation DataLoaders are built from the GroupKFold split.

Default split:

```text
train = all folds except fold 0
valid = fold 0
```

The training loader shuffles samples and drops the last incomplete batch. The validation loader keeps a deterministic order and evaluates all samples.


In [ ]:
train_df = df[df["fold"] != CFG.fold].reset_index(drop=True)
valid_df = df[df["fold"] == CFG.fold].reset_index(drop=True)

if CFG.smoke_test:
    # Small random subsample so the smoke test runs in minutes.
    # This only checks that the pipeline runs; it is not a real evaluation.
    train_df = train_df.sample(
        n=min(200, len(train_df)), random_state=CFG.seed
    ).reset_index(drop=True)
    valid_df = valid_df.sample(
        n=min(100, len(valid_df)), random_state=CFG.seed
    ).reset_index(drop=True)
    print(f"[SMOKE TEST] Subsampled -> train: {len(train_df)}, valid: {len(valid_df)}")

print("Train size:", len(train_df))
print("Valid size:", len(valid_df))

train_dataset = UWGIDataset(train_df, transform=get_train_transforms())
valid_dataset = UWGIDataset(valid_df, transform=get_valid_transforms())

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=False,
)


## 17. Build the U-Net Model

The segmentation model is created with `segmentation_models_pytorch`.

Key settings (driven by `CFG`):

```text
encoder_name    = CFG.encoder_name   (e.g. efficientnet-b0 / efficientnet-b5)
encoder_weights = imagenet
in_channels     = 3
classes         = 3
activation      = None
```

`activation=None` means the model outputs raw logits. Sigmoid activation is applied only when converting logits into probabilities for metrics or inference.

**Scaling the encoder:** switch `CFG.encoder_name` to a larger EfficientNet (b4/b5/...) for higher capacity. Larger encoders need more GPU memory, so pair them with a smaller `CFG.batch_size` and a larger `CFG.accum_steps` to keep the effective batch size stable.

**Optional classification head:** when `CFG.use_aux_cls = True`, `aux_params` adds a multi-label classification head that shares the encoder and predicts per-organ presence. The model then returns `(seg_logits, cls_logits)`; the helper `split_outputs` normalizes both cases so the rest of the pipeline is unchanged. These presence probabilities power the inference-time classification gate that can zero out false-positive masks.


In [ ]:
def build_model():
    aux_params = None
    if CFG.use_aux_cls:
        # Classification head shares the encoder with the U-Net decoder.
        # It outputs one logit per organ (presence / absence).
        aux_params = dict(
            pooling="avg",
            dropout=0.2,
            classes=len(CFG.classes),
            activation=None,
        )

    model = smp.Unet(
        encoder_name=CFG.encoder_name,
        encoder_weights=CFG.encoder_weights,
        in_channels=3,
        classes=len(CFG.classes),
        activation=None,
        aux_params=aux_params,
    )
    return model


def split_outputs(outputs):
    """
    Normalize model outputs into (seg_logits, cls_logits).

    With the aux head enabled, smp.Unet returns (masks, labels).
    Without it, the model returns a single segmentation tensor, so
    cls_logits is None.
    """
    if isinstance(outputs, (tuple, list)):
        seg_logits, cls_logits = outputs[0], outputs[1]
    else:
        seg_logits, cls_logits = outputs, None
    return seg_logits, cls_logits


model = build_model()
model = model.to(CFG.device)

num_params = sum(p.numel() for p in model.parameters())
num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Encoder: {CFG.encoder_name}")
print(f"Aux classification head: {CFG.use_aux_cls}")
print(f"Total parameters: {num_params:,}")
print(f"Trainable parameters: {num_trainable:,}")


## 18. Loss Function

The training objective combines pixel-level classification loss with overlap-based segmentation loss:

```text
Loss = BCEWithLogitsLoss + 3 × DiceLoss
```

BCEWithLogitsLoss handles per-pixel binary classification, while DiceLoss directly encourages better mask overlap. Both losses are configured to work with logits.

When the auxiliary classification head is enabled (`CFG.use_aux_cls = True`), a presence loss is added:

```text
Loss = seg_loss + cls_weight × BCEWithLogitsLoss(cls_logits, presence)
```

The presence target is derived on the fly from the masks (`masks_to_presence`): a class is present if its mask channel contains any positive pixel. This means no extra label columns or Dataset changes are required — the `has_*` columns are reproduced implicitly from the masks.


In [ ]:
bce_loss = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(
    mode="multilabel",
    from_logits=True,
    smooth=1e-6,
)

# Classification (presence) loss for the optional aux head.
cls_bce_loss = nn.BCEWithLogitsLoss()


def masks_to_presence(masks):
    """
    Derive per-class presence labels from masks.

    A class is present if its mask channel has any positive pixel.
    masks: [B, C, H, W] -> presence: [B, C] in {0., 1.}
    """
    return (masks.sum(dim=(2, 3)) > 0).float()


def criterion(outputs, masks):
    """
    Combined training loss.

    Segmentation: BCEWithLogits + dice_weight * Dice.
    If the aux classification head is active, add cls_weight * BCE on the
    presence labels derived from the masks.
    """
    seg_logits, cls_logits = split_outputs(outputs)

    bce = bce_loss(seg_logits, masks)
    dice = dice_loss(seg_logits, masks)
    loss = bce + CFG.dice_weight * dice

    if cls_logits is not None:
        cls_target = masks_to_presence(masks)
        loss = loss + CFG.cls_weight * cls_bce_loss(cls_logits, cls_target)

    return loss


## 19. Dice Metric

Validation performance is measured with Dice score.

The metric is reported as:

- mean Dice across the three organ classes
- per-class Dice for `large_bowel`, `small_bowel`, and `stomach`

Per-class reporting helps identify whether one organ is harder for the model to segment.


In [ ]:
def dice_coef_from_logits(logits, targets, threshold=0.5, eps=1e-7):
    """
    Calculate Dice score from logits.

    Parameters
    ----------
    logits : torch.Tensor
        Model outputs with shape [B, C, H, W].
    targets : torch.Tensor
        Ground-truth masks with shape [B, C, H, W].
    threshold : float
        Probability threshold for binary masks.
    eps : float
        Small value to avoid division by zero.

    Returns
    -------
    mean_dice : torch.Tensor
        Mean Dice over all classes.
    class_dice : torch.Tensor
        Dice score for each class.
    """
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    targets = targets.float()

    dims = (0, 2, 3)

    intersection = torch.sum(preds * targets, dim=dims)
    pred_sum = torch.sum(preds, dim=dims)
    target_sum = torch.sum(targets, dim=dims)

    class_dice = (2.0 * intersection + eps) / (pred_sum + target_sum + eps)

    # If prediction and target are both empty, define Dice as 1.
    empty_mask = (pred_sum + target_sum) == 0
    class_dice = torch.where(empty_mask, torch.ones_like(class_dice), class_dice)

    mean_dice = class_dice.mean()

    return mean_dice, class_dice


## 20. Optimizer, Scheduler, and AMP Scaler

The model is optimized with AdamW and a cosine annealing learning-rate schedule. Automatic mixed precision is enabled when CUDA is available to reduce memory usage and speed up training.

**Gradient accumulation:** when `CFG.accum_steps > 1`, gradients are accumulated over several batches before each optimizer step, so the effective batch size is `batch_size × accum_steps`. This lets a large encoder train at a small per-step `batch_size` (to fit in GPU memory) while keeping the optimization behavior of a larger batch. The optimizer steps every `accum_steps` batches and also on the final batch of each epoch.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG.epochs,
    eta_min=CFG.eta_min,
)

# Enable AMP only when CUDA is available.
use_amp = CFG.amp and (CFG.device == "cuda")

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp,
)

print("CUDA available:", torch.cuda.is_available())
print("Device:", CFG.device)
print("Use AMP:", use_amp)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 21. Training and Validation Functions

The training loop performs forward propagation, loss computation, backpropagation, optimizer updates, and metric logging.

The validation loop runs without gradient updates and reports mean Dice together with per-class Dice scores.


In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device, epoch, total_epochs):
    model.train()

    running_loss = 0.0
    running_dice = 0.0
    n_batches = 0

    accum_steps = max(1, getattr(CFG, "accum_steps", 1))

    pbar = tqdm(
        loader,
        total=len(loader),
        desc=f"Train Epoch {epoch:02d}/{total_epochs}",
        leave=False,
    )

    optimizer.zero_grad(set_to_none=True)

    for step, (images, masks) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(images)
            # Scale the loss so accumulated gradients match a full-size batch.
            loss = criterion(outputs, masks) / accum_steps

        scaler.scale(loss).backward()

        # Step the optimizer every accum_steps batches (and on the last batch).
        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        seg_logits, _ = split_outputs(outputs)

        with torch.no_grad():
            mean_dice, _ = dice_coef_from_logits(seg_logits, masks, threshold=0.5)

        # Report the unscaled loss for readability.
        running_loss += loss.item() * accum_steps
        running_dice += mean_dice.item()
        n_batches += 1

        avg_loss = running_loss / n_batches
        avg_dice = running_dice / n_batches
        current_lr = optimizer.param_groups[0]["lr"]

        pbar.set_postfix({
            "loss": f"{avg_loss:.4f}",
            "dice": f"{avg_dice:.4f}",
            "lr": f"{current_lr:.2e}",
        })

    epoch_loss = running_loss / max(n_batches, 1)
    epoch_dice = running_dice / max(n_batches, 1)

    return epoch_loss, epoch_dice


@torch.no_grad()
def valid_one_epoch(model, loader, device, epoch, total_epochs):
    model.eval()

    running_loss = 0.0
    running_dice = 0.0
    class_dice_sum = torch.zeros(len(CFG.classes), device=device)
    n_batches = 0

    pbar = tqdm(
        loader,
        total=len(loader),
        desc=f"Valid Epoch {epoch:02d}/{total_epochs}",
        leave=False,
    )

    for images, masks in pbar:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, masks)

        seg_logits, _ = split_outputs(outputs)
        mean_dice, class_dice = dice_coef_from_logits(seg_logits, masks, threshold=0.5)

        running_loss += loss.item()
        running_dice += mean_dice.item()
        class_dice_sum += class_dice
        n_batches += 1

        avg_loss = running_loss / n_batches
        avg_dice = running_dice / n_batches

        pbar.set_postfix({
            "loss": f"{avg_loss:.4f}",
            "dice": f"{avg_dice:.4f}",
        })

    epoch_loss = running_loss / max(n_batches, 1)
    epoch_dice = running_dice / max(n_batches, 1)
    epoch_class_dice = (class_dice_sum / max(n_batches, 1)).detach().cpu().numpy()

    return epoch_loss, epoch_dice, epoch_class_dice


## 22. Run Training

The model is trained for the configured number of epochs. After each epoch, validation Dice is computed and the best checkpoint is saved.

Output files:

```text
best_effnetb0_unet_c_order_fold0.pth
training_history_c_order_fold0.csv
```

The checkpoint stores the model state, optimizer state, scheduler state, best validation Dice, and core configuration values.


In [ ]:
best_dice = -1.0
history = []
start_epoch = 1

checkpoint_path = Path(f"./best_{CFG.model_tag}_unet_c_order_fold{CFG.fold}.pth")
last_path = Path(f"./last_{CFG.model_tag}_unet_c_order_fold{CFG.fold}.pth")
history_path = Path(f"./training_history_{CFG.model_tag}_c_order_fold{CFG.fold}.csv")


def build_state(epoch):
    """Full training state: weights + optimizer + scheduler + scaler + bookkeeping.

    Saved for both `best` (only on a new best Dice) and `last` (every epoch).
    The `last` checkpoint is what makes an interrupted run resumable: it lets a
    fresh Kaggle session pick up exactly where the previous one stopped.
    """
    return {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "epoch": epoch,
        "best_dice": best_dice,
        "cfg": {
            "encoder_name": CFG.encoder_name,
            "img_size": CFG.img_size,
            "slice_stride": CFG.slice_stride,
            "classes": CFG.classes,
            "fold": CFG.fold,
            "use_aux_cls": CFG.use_aux_cls,
            "cls_weight": CFG.cls_weight,
            "accum_steps": CFG.accum_steps,
        },
    }


# ------------------------------------------------------------------
# Resume: if a `last` checkpoint from a previous (interrupted) session
# exists, restore model / optimizer / scheduler / AMP scaler and continue
# from the next epoch. This is what lets a >12h training run finish across
# two or more Kaggle sessions instead of restarting from scratch.
# ------------------------------------------------------------------
if CFG.resume and last_path.exists():
    ckpt = torch.load(last_path, map_location=CFG.device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    best_dice = ckpt.get("best_dice", -1.0)
    start_epoch = ckpt["epoch"] + 1
    # rebuild the in-memory history (and keep the CSV continuous) up to the
    # last completed epoch, so the saved history doesn't get duplicated rows.
    if history_path.exists():
        prev = pd.read_csv(history_path).to_dict("records")
        history = [r for r in prev if r["epoch"] < start_epoch]
    print(
        f"Resuming from {last_path}: continue at epoch {start_epoch}, "
        f"best Dice so far {best_dice:.4f}"
    )

print("=" * 80)
print("Training started")
print("=" * 80)
print(f"Fold: {CFG.fold}")
print(f"Device: {CFG.device}")
print(f"Use AMP: {use_amp}")
print(f"Encoder: {CFG.encoder_name}")
print(f"Image size: {CFG.img_size}")
print(f"Batch size: {CFG.batch_size}")
print(f"Accum steps: {CFG.accum_steps} (effective batch = {CFG.batch_size * CFG.accum_steps})")
print(f"Aux classification head: {CFG.use_aux_cls}")
print(f"Epochs: {CFG.epochs} (starting at {start_epoch})")
print(f"Max train hours this session: {CFG.max_train_hours}")
print(f"Train samples: {len(train_dataset)}")
print(f"Valid samples: {len(valid_dataset)}")
print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Best checkpoint path: {checkpoint_path}")
print(f"Last checkpoint path: {last_path}")
print("=" * 80)

total_start_time = time.time()
stopped_early = False

for epoch in range(start_epoch, CFG.epochs + 1):
    epoch_start_time = time.time()

    print(f"\nEpoch {epoch}/{CFG.epochs} started")

    train_loss, train_dice = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        device=CFG.device,
        epoch=epoch,
        total_epochs=CFG.epochs,
    )

    valid_loss, valid_dice, valid_class_dice = valid_one_epoch(
        model=model,
        loader=valid_loader,
        device=CFG.device,
        epoch=epoch,
        total_epochs=CFG.epochs,
    )

    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    epoch_elapsed = time.time() - epoch_start_time

    row = {
        "epoch": epoch,
        "lr": lr,
        "train_loss": train_loss,
        "train_dice": train_dice,
        "valid_loss": valid_loss,
        "valid_dice": valid_dice,
        "epoch_time_sec": epoch_elapsed,
    }

    for i, c in enumerate(CFG.classes):
        row[f"valid_dice_{c}"] = valid_class_dice[i]

    history.append(row)

    print("-" * 80)
    print(f"Epoch {epoch}/{CFG.epochs} finished")
    print(f"Time: {epoch_elapsed:.1f} sec")
    print(f"LR: {lr:.2e}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Dice: {train_dice:.4f}")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Dice: {valid_dice:.4f}")

    print(
        "Per-class Valid Dice: "
        + " | ".join(
            [f"{c}: {valid_class_dice[i]:.4f}" for i, c in enumerate(CFG.classes)]
        )
    )

    # Update best_dice FIRST so the `last` checkpoint we write below records
    # the true best-so-far (otherwise a resumed run could regress best_dice).
    if valid_dice > best_dice:
        best_dice = valid_dice
        torch.save(build_state(epoch), checkpoint_path)
        print(f"New best checkpoint saved: {checkpoint_path}")
        print(f"Best Valid Dice: {best_dice:.4f}")

    # Always save `last` every epoch, so this session's progress is never lost.
    torch.save(build_state(epoch), last_path)

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

    # ------------------------------------------------------------------
    # Predictive wall-clock guard. If finishing one more epoch would likely
    # push us past the per-session budget, stop now -- cleanly, with `last`
    # and `best` already on disk -- instead of getting killed mid-epoch by
    # the Kaggle 12h limit (which in Commit mode loses ALL outputs).
    # ------------------------------------------------------------------
    elapsed = time.time() - total_start_time
    done_this_session = epoch - start_epoch + 1
    avg_epoch = elapsed / done_this_session
    budget = CFG.max_train_hours * 3600.0
    if epoch < CFG.epochs and elapsed + avg_epoch >= budget:
        stopped_early = True
        print("-" * 80)
        print(
            f"\nWall-clock budget reached: {elapsed / 3600:.2f}h used, "
            f"~{avg_epoch / 60:.1f} min/epoch, budget {CFG.max_train_hours}h.\n"
            f"Stopping cleanly after epoch {epoch}. Checkpoints saved:\n"
            f"  best -> {checkpoint_path}\n"
            f"  last -> {last_path}\n"
            f"Re-run this notebook (CFG.resume=True) to continue from epoch {epoch + 1}."
        )
        break

    print("-" * 80)

total_elapsed = time.time() - total_start_time

print("\n" + "=" * 80)
print("Training stopped early (budget)" if stopped_early else "Training finished")
print("=" * 80)
print(f"This session time: {total_elapsed / 60:.1f} min")
print(f"Last completed epoch: {epoch} / {CFG.epochs}")
print(f"Best valid Dice: {best_dice:.4f}")
print(f"Best checkpoint: {checkpoint_path}")
print(f"Last checkpoint: {last_path}")
print(f"History saved to: {history_path}")
if stopped_early:
    print("NOTE: training did not reach the final epoch -- re-run to resume.")
print("=" * 80)


## 23. Plot Training Curves

Training history is visualized with loss and Dice curves.

Useful patterns to check:

- Training loss decreases and validation Dice increases: normal learning behavior.
- Training loss decreases while validation Dice stays flat: possible overfitting or weak validation generalization.
- Both training and validation curves stay flat: possible issue with data, loss setup, or learning rate.


In [ ]:
history_df = pd.read_csv(history_path)

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
plt.plot(history_df["epoch"], history_df["valid_loss"], label="valid_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_dice"], label="train_dice")
plt.plot(history_df["epoch"], history_df["valid_dice"], label="valid_dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Dice Curve")
plt.legend()
plt.grid(True)
plt.show()


## 24. Visualize Validation Predictions

The best checkpoint is loaded and used to visualize validation predictions.

For each selected sample, the notebook displays:

- ground-truth masks
- predicted probability maps
- binary predictions after thresholding

The default visualization threshold is 0.5. Class-specific thresholds can be tuned separately during the inference stage.


In [ ]:
def load_best_model(checkpoint_path):
    model = build_model()
    checkpoint = torch.load(checkpoint_path, map_location=CFG.device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(CFG.device)
    model.eval()
    return model


best_model = load_best_model(checkpoint_path)


In [ ]:
@torch.no_grad()
def visualize_predictions_debug(model, dataset, indices=None, threshold=0.5):
    if indices is None:
        indices = [0, 1, 2]

    model.eval()

    for idx in indices:
        image, mask = dataset[idx]

        input_tensor = image.unsqueeze(0).to(CFG.device)

        outputs = model(input_tensor)
        seg_logits, cls_logits = split_outputs(outputs)

        prob = torch.sigmoid(seg_logits)[0].detach().cpu().numpy()
        pred = (prob > threshold).astype(np.float32)

        image_np = image.detach().cpu().numpy().transpose(1, 2, 0)
        mask_np = mask.detach().cpu().numpy()

        center_img = image_np[:, :, 1]

        print("=" * 80)
        print(f"Validation sample index: {idx}")
        print(f"Threshold: {threshold}")

        # When the aux head is active, also show the predicted presence
        # probability per organ (the signal used by the classification gate).
        if cls_logits is not None:
            cls_prob = torch.sigmoid(cls_logits)[0].detach().cpu().numpy()
            print(
                "Presence prob: "
                + " | ".join(
                    [f"{c}: {cls_prob[i]:.3f}" for i, c in enumerate(CFG.classes)]
                )
            )

        for i, c in enumerate(CFG.classes):
            print(
                f"{c}: "
                f"GT area={mask_np[i].sum():.0f}, "
                f"max_prob={prob[i].max():.4f}, "
                f"pred_area={pred[i].sum():.0f}"
            )

        fig, axes = plt.subplots(3, 3, figsize=(14, 12))

        for i, c in enumerate(CFG.classes):
            axes[0, i].imshow(center_img, cmap="gray")
            axes[0, i].imshow(mask_np[i], cmap="Reds", alpha=0.5)
            axes[0, i].set_title(f"Ground Truth: {c}")
            axes[0, i].axis("off")

            axes[1, i].imshow(center_img, cmap="gray")
            im = axes[1, i].imshow(prob[i], cmap="jet", alpha=0.45, vmin=0, vmax=1)
            axes[1, i].set_title(f"Probability: {c}, max={prob[i].max():.3f}")
            axes[1, i].axis("off")

            axes[2, i].imshow(center_img, cmap="gray")
            axes[2, i].imshow(pred[i], cmap="Reds", alpha=0.5)
            axes[2, i].set_title(f"Prediction: {c}, area={pred[i].sum():.0f}")
            axes[2, i].axis("off")

        plt.tight_layout()
        plt.show()

valid_vis_dataset = UWGIDataset(valid_df, transform=get_valid_transforms())
visualize_predictions_debug(best_model, valid_vis_dataset, indices=[0, 1, 2], threshold=0.5)


## Next Steps

Possible extensions after this training pipeline include:

1. Tune class-specific probability thresholds on the validation set.
2. Add horizontal flip test-time augmentation for inference.
3. Generate `submission.csv` from test predictions.
4. Train additional folds for cross-validation.
5. Average predictions from multiple folds or model variants.
